In [2]:

import json
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import (
    log_loss,
    brier_score_loss,
    roc_auc_score,
)

from cricketwp.predict import chase_curve


# ============================================================
# 1. LOCATE PROJECT ROOT
# ============================================================

ROOT = Path.cwd()

# Notebook may be running from the notebooks/ directory
if not (ROOT / "data" / "processed").exists():
    ROOT = ROOT.parent

assert (ROOT / "data" / "processed" / "chase_test.parquet").exists(), (
    f"Could not find chase_test.parquet. Detected root: {ROOT}"
)

print("PROJECT ROOT:", ROOT)


# ============================================================
# 2. LOAD DATASETS
# ============================================================

train = pd.read_parquet(
    ROOT / "data" / "processed" / "chase_train.parquet"
)

te = pd.read_parquet(
    ROOT / "data" / "processed" / "chase_test.parquet"
)

print("\nDATASET INFORMATION")
print("Training rows:", len(train))
print("Test rows:", len(te))
print("Training matches:", train["match_id"].nunique())
print("Test matches:", te["match_id"].nunique())


# ============================================================
# 3. LOAD PRODUCTION MODEL
# ============================================================

clf = xgb.XGBClassifier()

clf.load_model(
    str(ROOT / "artifacts" / "chase_xgb.json")
)

with open(
    ROOT / "artifacts" / "chase_iso.json",
    encoding="utf-8",
) as f:
    iso = json.load(f)

iso_x = np.asarray(iso["x"])
iso_y = np.asarray(iso["y"])

with open(
    ROOT / "artifacts" / "meta.json",
    encoding="utf-8",
) as f:
    meta = json.load(f)

print("\nMODEL VERSION:", meta["version"])


# ============================================================
# 4. REPRODUCE PRODUCTION TEST PREDICTIONS
# ============================================================

curve = chase_curve(
    te,
    clf,
    iso_x,
    iso_y,
)

p_test = curve["p_chase"].to_numpy()
y_test = te["label"].to_numpy()

assert len(p_test) == len(y_test)
assert np.isfinite(p_test).all()
assert np.isin(y_test, [0, 1]).all()

print("\nPREDICTION CHECKS")
print("Predictions:", len(p_test))
print("Minimum probability:", p_test.min())
print("Maximum probability:", p_test.max())
print("NaN predictions:", np.isnan(p_test).sum())


# ============================================================
# 5. RECOMPUTE TEST METRICS
# ============================================================

reproduced = {
    "logloss": log_loss(
        y_test,
        p_test,
        labels=[0, 1],
    ),
    "brier": brier_score_loss(
        y_test,
        p_test,
    ),
    "auc": roc_auc_score(
        y_test,
        p_test,
    ),
}

print("\n" + "=" * 65)
print("REPRODUCED MODEL TEST METRICS")
print("=" * 65)

for name, value in reproduced.items():
    print(f"{name:12s}: {value:.8f}")


# ============================================================
# 6. COMPARE AGAINST SAVED METRICS
# ============================================================

saved = meta["test_metrics"]

print("\n" + "=" * 65)
print("SAVED VS REPRODUCED METRICS")
print("=" * 65)

for name, value in reproduced.items():
    original = saved[name]
    difference = abs(value - original)

    print(
        f"{name:12s} | "
        f"Saved: {original:.8f} | "
        f"Reproduced: {value:.8f} | "
        f"Difference: {difference:.10f}"
    )


# ============================================================
# 7. PREPARE BASELINE REFERENCE
# ============================================================

# Simple baseline:
# Always predict the chase-win frequency observed in training.

baseline_probability = float(train["label"].mean())

baseline_predictions = np.full(
    len(y_test),
    baseline_probability,
)

baseline_scores = {
    "logloss": log_loss(
        y_test,
        baseline_predictions,
        labels=[0, 1],
    ),
    "brier": brier_score_loss(
        y_test,
        baseline_predictions,
    ),
    "auc": 0.5,
}

print("\n" + "=" * 65)
print("SIMPLE BASELINE")
print("=" * 65)

print(
    "Training chase-win frequency:",
    round(baseline_probability, 6),
)

for name, value in baseline_scores.items():
    print(f"{name:12s}: {value:.8f}")


# ============================================================
# 8. FINAL EVALUATION SUMMARY
# ============================================================

print("\n" + "=" * 65)
print("TRUST EVALUATION SUMMARY")
print("=" * 65)

print("Test rows:", len(te))
print("Unique test matches:", te["match_id"].nunique())
print("Actual chase-win rate:", round(float(y_test.mean()), 6))
print("Baseline probability:", round(baseline_probability, 6))

print("\nMetric reproduction status:")

for name, value in reproduced.items():
    difference = abs(value - saved[name])
    status = "MATCH" if difference < 1e-5 else "CHECK"

    print(f"{name:12s}: {status}")

print("\nStep 1 complete.")


PROJECT ROOT: c:\Users\karne_dvob0bk\cric-win-pred

DATASET INFORMATION
Training rows: 302180
Test rows: 114440
Training matches: 2811
Test matches: 1133

MODEL VERSION: 1.0.0

PREDICTION CHECKS
Predictions: 114440
Minimum probability: 0.0
Maximum probability: 1.0
NaN predictions: 0

REPRODUCED MODEL TEST METRICS
logloss     : 0.32540718
brier       : 0.10287301
auc         : 0.93767979

SAVED VS REPRODUCED METRICS
logloss      | Saved: 0.32171693 | Reproduced: 0.32540718 | Difference: 0.0036902432
brier        | Saved: 0.10287301 | Reproduced: 0.10287301 | Difference: 0.0000000003
auc          | Saved: 0.93767979 | Reproduced: 0.93767979 | Difference: 0.0000000000

SIMPLE BASELINE
Training chase-win frequency: 0.491432
logloss     : 0.69233922
brier       : 0.24959606
auc         : 0.50000000

TRUST EVALUATION SUMMARY
Test rows: 114440
Unique test matches: 1133
Actual chase-win rate: 0.472143
Baseline probability: 0.491432

Metric reproduction status:
logloss     : CHECK
brier       :

In [3]:
# ============================================================
# LOG-LOSS DISCREPANCY INVESTIGATION
# ============================================================

print("SAVED LOG LOSS:", meta["test_metrics"]["logloss"])

print("\nEXACT EXTREME PROBABILITIES")

print("p = 0:", np.sum(p_test == 0))
print("p = 1:", np.sum(p_test == 1))

print(
    "Incorrect p = 0 predictions:",
    np.sum((p_test == 0) & (y_test == 1)),
)

print(
    "Incorrect p = 1 predictions:",
    np.sum((p_test == 1) & (y_test == 0)),
)

print("\nLOG-LOSS UNDER DIFFERENT CLIPPING VALUES")

for eps in [1e-3, 1e-5, 1e-7, 1e-10, 1e-15]:
    clipped = np.clip(p_test, eps, 1 - eps)

    score = log_loss(
        y_test,
        clipped,
        labels=[0, 1],
    )

    print(f"eps={eps:<10} logloss={score:.8f}")


# ============================================================
# BASELINE RESULTS
# ============================================================

print("\nBASELINE METRICS")

for name, value in baseline_scores.items():
    print(f"{name:12s}: {value:.8f}")

SAVED LOG LOSS: 0.32171693495108317

EXACT EXTREME PROBABILITIES
p = 0: 16833
p = 1: 6469
Incorrect p = 0 predictions: 4
Incorrect p = 1 predictions: 15

LOG-LOSS UNDER DIFFERENT CLIPPING VALUES
eps=0.001      logloss=0.32077343
eps=1e-05      logloss=0.32133648
eps=1e-07      logloss=0.32209904
eps=1e-10      logloss=0.32324589
eps=1e-15      logloss=0.32515743

BASELINE METRICS
logloss     : 0.69233922
brier       : 0.24959606
auc         : 0.50000000


In [4]:
# ============================================================
# COMPARE PRE-TERMINAL AND PRODUCTION PROBABILITIES
# ============================================================

# Raw XGBoost probabilities
raw_test = clf.predict_proba(te[__import__(
    "cricketwp.features",
    fromlist=["CHASE_FEATURES"]
).CHASE_FEATURES])[:, 1]

# Isotonic-calibrated probabilities BEFORE terminal rules
p_calibrated = np.interp(raw_test, iso_x, iso_y)

# Production probabilities AFTER terminal rules
p_production = p_test

# Compare metrics
print("=" * 65)
print("LOG-LOSS INVESTIGATION")
print("=" * 65)

for name, probabilities in [
    ("Raw XGBoost", raw_test),
    ("Isotonic calibrated", p_calibrated),
    ("Production (terminal rules)", p_production),
]:
    print(f"\n{name}")
    print("Log loss:", log_loss(y_test, probabilities, labels=[0, 1]))
    print("Brier:", brier_score_loss(y_test, probabilities))
    print("AUC:", roc_auc_score(y_test, probabilities))
    print("Exact zeros:", np.sum(probabilities == 0))
    print("Exact ones:", np.sum(probabilities == 1))


# ============================================================
# IDENTIFY TERMINAL RULE EFFECT
# ============================================================

changed = ~np.isclose(
    p_calibrated,
    p_production,
    atol=0,
    rtol=0,
)

print("\n" + "=" * 65)
print("TERMINAL RULE IMPACT")
print("=" * 65)

print("Rows changed by terminal rules:", changed.sum())

print(
    "Incorrect predictions among changed rows:",
    np.sum((p_production[changed] != y_test[changed]))
)

print(
    "Saved log loss:",
    meta["test_metrics"]["logloss"],
)

LOG-LOSS INVESTIGATION

Raw XGBoost
Log loss: 0.32087546586990356
Brier: 0.10162685811519623
AUC: 0.9377810022424455
Exact zeros: 0
Exact ones: 0

Isotonic calibrated
Log loss: 0.32559434489228156
Brier: 0.10290673897060984
AUC: 0.9376260397991222
Exact zeros: 16754
Exact ones: 6225

Production (terminal rules)
Log loss: 0.32540717814210646
Brier: 0.10287300958688492
AUC: 0.9376797880767507
Exact zeros: 16833
Exact ones: 6469

TERMINAL RULE IMPACT
Rows changed by terminal rules: 323
Incorrect predictions among changed rows: 0
Saved log loss: 0.32171693495108317


In [6]:
from cricketwp.eval import evaluate

print("SAVED META METRICS")
print(meta["test_metrics"])

print("\nPRODUCTION PREDICTIONS — ORIGINAL EVALUATOR")
production_scores = evaluate(y_test, p_test)

for metric, value in production_scores.items():
    saved = meta["test_metrics"].get(metric)
    difference = None if saved is None else value - saved

    print(
        f"{metric:10s} "
        f"reproduced={value:.10f} "
        f"saved={saved} "
        f"difference={difference}"
    )

print("\nRAW XGBOOST + TERMINAL RULES — ORIGINAL EVALUATOR")
raw_scores = evaluate(y_test, p_raw_production)

for metric, value in raw_scores.items():
    print(f"{metric:10s} {value:.10f}")

SAVED META METRICS
{'logloss': 0.32171693495108317, 'brier': 0.10287300925503678, 'auc': 0.9376797880767507, 'ece': 0.03188359776016573}

PRODUCTION PREDICTIONS — ORIGINAL EVALUATOR
logloss    reproduced=0.3217169350 saved=0.32171693495108317 difference=0.0
brier      reproduced=0.1028730093 saved=0.10287300925503678 difference=0.0
auc        reproduced=0.9376797881 saved=0.9376797880767507 difference=0.0
ece        reproduced=0.0318835978 saved=0.03188359776016573 difference=0.0

RAW XGBOOST + TERMINAL RULES — ORIGINAL EVALUATOR
logloss    0.3205960095
brier      0.1015898660
auc        0.9378347782
ece        0.0267246944


In [7]:
# Build 10 approximately equal-sized probability groups
t = pd.DataFrame({
    "p": p_test,
    "y": y_test
})

t["bin"] = pd.qcut(
    t["p"],
    q=10,
    duplicates="drop"
)

g = (
    t.groupby("bin", observed=True)
    .agg(
        pred=("p", "mean"),
        actual=("y", "mean"),
        n=("y", "size")
    )
    .reset_index()
)

# Remove pandas Interval objects; retain chart-ready values
trust_bins = [
    {
        "pred": round(float(row.pred), 6),
        "actual": round(float(row.actual), 6),
        "n": int(row.n)
    }
    for row in g.itertuples()
]

print("RELIABILITY BINS")
for row in trust_bins:
    print(row)

print("\nTOTAL STATES:", sum(row["n"] for row in trust_bins))
print("TOTAL MATCHES:", int(te["match_id"].nunique()))
print("NUMBER OF BINS:", len(trust_bins))

RELIABILITY BINS
{'pred': 0.005344, 'actual': 0.003003, 'n': 22976}
{'pred': 0.074377, 'actual': 0.057724, 'n': 11555}
{'pred': 0.16644, 'actual': 0.186331, 'n': 13417}
{'pred': 0.302443, 'actual': 0.369729, 'n': 9296}
{'pred': 0.469509, 'actual': 0.574116, 'n': 12440}
{'pred': 0.70489, 'actual': 0.772608, 'n': 13712}
{'pred': 0.857832, 'actual': 0.888943, 'n': 8158}
{'pred': 0.960074, 'actual': 0.963926, 'n': 13112}
{'pred': 0.997312, 'actual': 0.995703, 'n': 9774}

TOTAL STATES: 114440
TOTAL MATCHES: 1133
NUMBER OF BINS: 9


In [8]:
import json

from cricketwp.eval import evaluate

# 1. Evaluate the production model using the original evaluator
model_scores = evaluate(y_test, p_test)

# 2. Build a simple baseline from training-set win frequency
baseline_probability = float(train["label"].mean())

baseline_predictions = np.full(
    len(y_test),
    baseline_probability
)

baseline_scores = evaluate(y_test, baseline_predictions)

# 3. Assemble the trust report
trust = {
    "version": meta["version"],
    "evaluation_date": meta["date"],
    "metrics": model_scores,
    "baseline": baseline_scores,
    "baseline_definition": (
        "Constant probability equal to the training-set chase-win frequency"
    ),
    "bins": trust_bins,
    "n_states": int(len(te)),
    "n_matches": int(te["match_id"].nunique()),
    "known_limitations": [
        "Metrics are calculated over ball-by-ball match states, not independent matches.",
        "States from the same match are correlated.",
        "Reliability bins use quantile grouping; duplicate probability boundaries reduce the number of bins.",
        "Shortened-innings handling limitation F-01 remains unresolved."
    ]
}

# 4. Save the artifact
trust_path = ROOT / "artifacts" / "trust.json"
trust_path.parent.mkdir(parents=True, exist_ok=True)

with open(trust_path, "w", encoding="utf-8") as f:
    json.dump(trust, f, indent=2)

# 5. Read it back and verify
with open(trust_path, encoding="utf-8") as f:
    verified_trust = json.load(f)

print("SAVED:", trust_path)
print("FILE SIZE:", trust_path.stat().st_size, "bytes")
print("MODEL METRICS:", verified_trust["metrics"])
print("BASELINE METRICS:", verified_trust["baseline"])
print("RELIABILITY BINS:", len(verified_trust["bins"]))
print("TEST STATES:", verified_trust["n_states"])
print("TEST MATCHES:", verified_trust["n_matches"])

SAVED: c:\Users\karne_dvob0bk\cric-win-pred\artifacts\trust.json
FILE SIZE: 1632 bytes
MODEL METRICS: {'logloss': 0.32171693495108317, 'brier': 0.10287300925503678, 'auc': 0.9376797880767507, 'ece': 0.03188359776016573}
BASELINE METRICS: {'logloss': 0.6923392213296289, 'brier': 0.24959605633460477, 'auc': 0.5, 'ece': 0.019289651438623323}
RELIABILITY BINS: 9
TEST STATES: 114440
TEST MATCHES: 1133
